# 02. Preprocessing & Feature Engineering Pipeline
## Academic Project: Autonomous Warehouse AI — Predictive Analytics Component

### Objective
Raw warehouse attributes (e.g. `stock_level`, `daily_demand`, `lead_time_days`) alone lack dynamic context. This notebook develops **9 domain-specific operational features**:
1. **Days of Supply**: $\frac{\text{stock\_level}}{\text{daily\_demand} + \epsilon}$ (runout horizon)
2. **Lead Time Demand**: $\text{daily\_demand} \times \text{lead\_time\_days}$
3. **Replenish Cycle Demand**: $\text{daily\_demand} \times (\text{lead\_time\_days} + \text{reorder\_frequency\_days})$
4. **Safety Stock Coverage**: $\frac{\text{stock\_level}}{1.65 \times \text{demand\_std\_dev} \times \sqrt{\text{lead\_time}} + 1.0}$
5. **Reorder Buffer Ratio**: $\frac{\text{stock\_level}}{\text{reorder\_point} + 1.0}$
6. **Carrying-to-Handling Ratio**: $\frac{30 \times \text{holding\_cost}}{\text{handling\_cost} + \epsilon}$
7. **Stockout Pressure Index**: $\frac{\text{stockout\_count\_last\_month} + 1}{\text{fulfillment\_rate} + 1}$
8. **Turnover Velocity**: $\frac{\text{turnover\_ratio} \times \text{daily\_demand}}{\text{stock\_level} + 1}$
9. **Picking Friction Index**: $\frac{\text{picking\_time}}{\text{layout\_efficiency} + \epsilon}$

Plus temporal calendar features (`restock_month`, `restock_dayofweek`, `days_since_last_restock`).

Strict protocol:
- Stratified 70% Train, 15% Validation, 15% Test split (`seed=42`).
- Encoders and scalers fitted on Train only.

In [ ]:
import os
import sys
import joblib
import pandas as pd
import numpy as np

sys.path.append(os.path.abspath("../src"))
from preprocessing import build_preprocessing_pipeline, engineer_features
from data_loader import get_prepared_dataframe

print("Preprocessing modules loaded.")

### 1. Execute Feature Engineering and Stratified Splitting

In [ ]:
splits = build_preprocessing_pipeline()

X_train, X_val, X_test = splits["X_train"], splits["X_val"], splits["X_test"]
y_train, y_val, y_test = splits["y_train"], splits["y_val"], splits["y_test"]

print(f"Train samples:      {len(X_train)} ({len(X_train)/(len(X_train)+len(X_val)+len(X_test))*100:.1f}%) | High-risk: {y_train.sum()} ({y_train.mean()*100:.1f}%)")
print(f"Validation samples: {len(X_val)}   ({len(X_val)/(len(X_train)+len(X_val)+len(X_test))*100:.1f}%) | High-risk: {y_val.sum()} ({y_val.mean()*100:.1f}%)")
print(f"Test samples:       {len(X_test)}  ({len(X_test)/(len(X_train)+len(X_val)+len(X_test))*100:.1f}%) | High-risk: {y_test.sum()} ({y_test.mean()*100:.1f}%)")
print(f"Total Feature Dimensions: {X_train.shape[1]}")

### 2. Feature Dimension Inspection

In [ ]:
print("Engineered and One-Hot Encoded Feature List:")
for idx, col in enumerate(splits["feature_names"]):
    print(f"{idx+1:>2}. {col}")

### 3. Verification of Pipeline Artifacts

In [ ]:
pipeline = joblib.load("../data/processed/pipeline.pkl")
print("Pipeline components:", list(pipeline.keys()))
print("Categorical columns encoded:", pipeline["cat_cols"])
print("OneHot categories count:", len(pipeline["encoded_cat_names"]))